# Imports

In [1]:
from google.colab import files
import io
import os

import numpy as np
import pandas as pd

from sklearn.utils import shuffle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras

# Dataset

Load the pre-processed dataset.

In [2]:
# Remove file from colab environment if it already exists, to prevent multiple uploads when running code cell
!rm -rf HDB_Resale_Train.csv
!rm -rf HDB_Resale_Test.csv

# Create dictionary and store the data files
uploaded = files.upload()

# List all files and directories in the current working directory
print(os.listdir('.'))

# List files in uploaded
print(uploaded.keys())

# Train and test set
Train_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Train.csv']))
Test_df = pd.read_csv(io.BytesIO(uploaded['HDB_Resale_Test.csv']))

Saving HDB_Resale_Train.csv to HDB_Resale_Train.csv
Saving HDB_Resale_Test.csv to HDB_Resale_Test.csv
['.config', 'HDB_Resale_Train.csv', 'HDB_Resale_Test.csv', 'sample_data']
dict_keys(['HDB_Resale_Train.csv', 'HDB_Resale_Test.csv'])


### Additional Dataset Preparation

Unlike the global and local single-task approaches, the preparation of the dataset required for the MTL model is much more complicated.

Steps:
1. 'Town' feature must be kept in one column, and it will not be used as a feature in training. It will be used to split the output into regions.
2. The rest of the categorical columns are encoded and the continuous columns are scaled.
3. X_train and X_test contain features from the full dataset, excluding 'town'
4. y_train and y_test have to be reformatted to a dictionary of arrays, each containing the sample's actual value and 4 placeholder values for the other regions, e.g. an impossible value like -$1.
5. Finally, a custom loss function is also required to detect the placeholder values and ignore them.   

### Encoding Categorical Features
For 'month_sold' feature:
- Perform cyclical encoding on the new column, which splits the column into two columns using sine and cosine:
    - sin(2π * month/12)
    - cos(2π * month/12)
- This models the feature in a cycle, meaning that 12 is as close to 1 as is to 11. This preserves the meaning of the month column.

For the rest of the categorical features except 'town':
- Perform one-hot-encoding

In [3]:
Train_df_encoded = Train_df.copy()

# Apply cyclical encoding to 'month_sold' feature
Train_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * Train_df['month_sold']/12)
Train_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * Train_df['month_sold']/12)

# Drop original month sold feature
Train_df_encoded = Train_df_encoded.drop(['month_sold'], axis=1)

categorical_cols = ['flat_type', 'storey_range', 'flat_model']

# Apply one-hot-encoding to the remaining catergorical features, with dtype=int for 0s and 1s
Train_df_encoded_final = pd.get_dummies(Train_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

Test_df_encoded = Test_df.copy()

## Apply cyclical encoding to 'month_sold' feature
Test_df_encoded['month_sold_sin'] = np.sin(2 * np.pi * Test_df['month_sold']/12)
Test_df_encoded['month_sold_cos'] = np.cos(2 * np.pi * Test_df['month_sold']/12)

# Drop original month sold feature
Test_df_encoded = Test_df_encoded.drop(['month_sold'], axis=1)

# Apply one-hot-encoding to the remaining catergorical features, with dtype=int for 0s and 1s
Test_df_encoded_final = pd.get_dummies(Test_df_encoded, columns=categorical_cols, drop_first=False, dtype=int)

# Align columns for consistent feature sets between training and testing
# This ensures that if a category is present in training but not testing or vice versa,
# the dataFrames still have all columns, with missing ones filled with 0s.
aligned_columns = list(Train_df_encoded_final.columns)
Test_df_encoded_final = Test_df_encoded_final.reindex(columns=aligned_columns, fill_value=0)

### Scaling Continuous Features

Scale the continuous features using StandardScaler(), ensuring the test set is transformed using the same scaler used for the training set.

In [4]:
# Define the continuous features to scale
continuous_cols = ['floor_area_sqm', 'resale_price', 'months_since_jan_2017', 'remaining_lease_years']

# Initialize the StandardScaler
scaler = StandardScaler()

Train_df_final = Train_df_encoded_final.copy()
Test_df_final = Test_df_encoded_final.copy()

# Fit the scaler on the training data's continuous features and transform them
Train_df_final[continuous_cols] = scaler.fit_transform(Train_df_final[continuous_cols])

# Transform the testing data's continuous features using the fitted scaler
Test_df_final[continuous_cols] = scaler.transform(Test_df_final[continuous_cols])

# Multi-Task Learning (MTL) Approach

MTL is similar to local single-task approach, except only one model is trained, and the output is split into the different tasks. (e.g. 5 output layers in a neural network) Individual error metric values must be used to calculate the weighted error metric before comparison can be done.

Task selection: We have decided to use 'Town' feature to divide the dataset into regions for this baseline. The model we will be using is a Multi-Output Neural Network.

---

Neural networks are naturally stochastic, meaning certain operations are always random, such as:
- Model Weight Initialisation: Initial weights assigned to neurons are random.
- Dropout Layers: A set percentage but neurons chosen are random.
- Batches trained: The order is random.

Because of this, the model's results are different each time. By using a fixed random seed, the operations will be randomised in the same way each time. This will result in improved reproducibility (though results will still vary slightly).

---

The performance metrics used are:
1. Mean Squared Error (MSE) - Used for the loss function. MSE prioritises reducing the difference between predicted and actual values, so it this the best metric for house price prediction.
3. Root Mean Squared Error (RMSE) - Used as a performance metric to allow the results to be easily interpretable, it presents the loss in the same units as the target. However since the target variable was scaled, RMSE must be unscaled to be interpretable.
4. R-squared - Used as a performance metric to allow the results to be easily interpretable, it presents the percentage of variance explained by the model.

In [5]:
# Shuffle the training data to ensure the data is not in chronological order
Train_df_final = shuffle(Train_df_final, random_state=42)

# Reset index
Train_df_final = Train_df_final.reset_index(drop=True)

### Custom y-train and y_test

In [6]:
# Define regions and their corresponding towns
regions = {
    'North': ['PUNGGOL', 'SEMBAWANG', 'SENGKANG', 'WOODLANDS', 'YISHUN'],
    'South': ['BUKIT MERAH', 'GEYLANG', 'MARINE PARADE', 'QUEENSTOWN'],
    'East': ['BEDOK', 'HOUGANG', 'PASIR RIS', 'TAMPINES'],
    'West': ['BUKIT BATOK', 'BUKIT PANJANG', 'CHOA CHU KANG', 'CLEMENTI', 'JURONG EAST', 'JURONG WEST'],
    'Central': ['ANG MO KIO', 'BISHAN', 'BUKIT TIMAH', 'CENTRAL AREA', 'KALLANG/WHAMPOA', 'SERANGOON', 'TOA PAYOH']
}

In [7]:
# Get unique regions (tasks) from the defined regions dictionary
region_names = sorted(regions.keys())
num_regions = len(region_names)

# Create a mapping from town to region for easier lookup
town_to_region = {}
for region, towns_list in regions.items():
    for town in towns_list:
        town_to_region[town] = region

# Placeholder for ignored values in y_train and y_test
PLACEHOLDER_VALUE = -1.0

# Create y_train_mtl as a dictionary of arrays, with regions as keys
y_train_mtl = {}
for region in region_names:
    # Initialise array for each region with placeholder values
    y_train_mtl[f'output_{region}'] = np.full((len(Train_df_final), 1), PLACEHOLDER_VALUE, dtype=np.float32)

# Populate y_train_mtl with actual resale prices, mapping town to region
for i in range(len(Train_df_final)):
    current_town = Train_df_final.loc[i, 'town']
    current_price = Train_df_final.loc[i, 'resale_price']
    current_region = town_to_region.get(current_town) # Get the region for the current town
    if current_region:
        y_train_mtl[f'output_{current_region}'][i, 0] = current_price

# Create y_test_mtl similarly for consistent loss calculation during evaluation
y_test_mtl = {}
for region in region_names:
    y_test_mtl[f'output_{region}'] = np.full((len(Test_df_final), 1), PLACEHOLDER_VALUE, dtype=np.float32)

for i in range(len(Test_df_final)):
    current_town = Test_df_final.loc[i, 'town']
    current_price = Test_df_final.loc[i, 'resale_price']
    current_region = town_to_region.get(current_town)
    if current_region:
        y_test_mtl[f'output_{current_region}'][i, 0] = current_price

# One-hot-encode 'town' now that y_train and y_test are mapped
Train_df_final = pd.get_dummies(Train_df_final, columns=['town'], drop_first=False, dtype=int)
Test_df_final = pd.get_dummies(Test_df_final, columns=['town'], drop_first=False, dtype=int)
aligned_columns = list(Train_df_final.columns)
Test_df_final = Test_df_final.reindex(columns=aligned_columns, fill_value=0)

# Define X_train and X_test
X_train = Train_df_final.drop(columns=['resale_price'])
X_test = Test_df_final.drop(columns=['resale_price'])

print(f"Number of unique regions (tasks): {num_regions}")
print(f"y_train_mtl keys: {list(y_train_mtl.keys())}")

Number of unique regions (tasks): 5
y_train_mtl keys: ['output_Central', 'output_East', 'output_North', 'output_South', 'output_West']


### Custom MSE Loss Function

Since y_train and y_test contain placeholder values for regions that are not relevant to a specific sample, a standard Mean Squared Error (MSE) loss function would incorrectly penalise these placeholders. To address this, a custom loss function is defined. This function filters out the placeholder values before calculating the MSE, ensuring that the model only learns from the actual resale_price values.

In [8]:
def mtl_mse_loss(y_true, y_pred):
    # Cast y_true to float32 to ensure compatibility
    y_true = tf.cast(y_true, tf.float32)

    # Create a mask for valid (non-placeholder) values
    valid_mask = tf.not_equal(y_true, PLACEHOLDER_VALUE)
    valid_mask = tf.cast(valid_mask, tf.float32) # Convert boolean mask to float for multiplication

    # Calculate squared difference
    squared_difference = tf.square(y_true - y_pred)

    # Apply the mask: placeholder values will have 0 loss contribution
    masked_squared_difference = squared_difference * valid_mask

    # Sum of valid squared differences
    sum_valid_squared_difference = tf.reduce_sum(masked_squared_difference)

    # Count of valid elements
    num_valid_elements = tf.reduce_sum(valid_mask)

    # Avoid division by zero if there are no valid elements
    loss = tf.cond(tf.greater(num_valid_elements, 0),
                   lambda: sum_valid_squared_difference / num_valid_elements,
                   lambda: 0.0)
    return loss

### Initial Model

The model is compiled with the Adam optimizer and the custom `mtl_mse_loss` applied to each output head. Mean Squared Error (MSE) is used as a metric for each output during training.


Layers:
- Input layer
- 3 shared Dense layers
    - Neurons: 128 > 64 > 32
    - Activation: reLu
- 2 Dropout layers to prevent overfitting (Deactivates 20% of the neurons)
- 5 dedicated output layers
    - Neurons: 1
    - Activation: linear

Parameters:
- Learning rate: 0.001 (default for adam optimiser)
- Loss function: Custom MSE
- Epochs: 20 (Smaller value to prevent overfitting, since there is no checkpoint)
- Batch size: 32
- Validation set: 20% of training data
- Checkpoint: No checkpoint implemented as there are 5 outputs with different best weights, so it would be biased to choose the best weight for one region to use as the checkpoint.

In [19]:
# Define input layer
input_tensor = keras.Input(shape=(X_train.shape[1],))

# Shared base layers
x = keras.layers.Dense(128, activation='relu')(input_tensor)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(32, activation='relu')(x)

# Create multiple output heads, one for each region
output_tensors = []
# Iterate over region_names
for region in region_names:
    # Each output head is a single dense layer with 1 neuron and 'linear' activation (for regression)
    output_head = keras.layers.Dense(1, activation='linear', name=f'output_{region}')(x)
    output_tensors.append(output_head)

# Create the MTL model
model = keras.Model(inputs=input_tensor, outputs=output_tensors)

# Compile the model
# Need a dictionary of losses, one for each output, using region names
loss_dict = {f'output_{region}': mtl_mse_loss for region in region_names}

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss=loss_dict)

model.summary()

# Train the model
history = model.fit(X_train, y_train_mtl,
                    epochs=20,
                    batch_size=32,
                    validation_split=0.2,
                    verbose=1)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 76)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │      9,856 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 64)        │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │      2,080 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_Central      │ (None, 1)         │         33 │ dense_8[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_East (Dense) │ (None, 1)         │         33 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_North        │ (None, 1)         │         33 │ dense_8[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_South        │ (None, 1)         │         33 │ dense_8[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_West (Dense) │ (None, 1)         │         33 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,357 (79.52 KB)

 Trainable params: 20,357 (79.52 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 25s 4ms/step - loss: 0.5724 - output_Central_loss: 0.1463 - output_East_loss: 0.0884 - output_North_loss: 0.0801 - output_South_loss: 0.1524 - output_West_loss: 0.1052 - val_loss: 0.3132 - val_output_Central_loss: 0.0818 - val_output_East_loss: 0.0509 - val_output_North_loss: 0.0416 - val_output_South_loss: 0.0798 - val_output_West_loss: 0.0591
Epoch 2/20
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 22s 4ms/step - loss: 0.3579 - output_Central_loss: 0.0947 - output_East_loss: 0.0556 - output_North_loss: 0.0491 - output_South_loss: 0.0923 - output_West_loss: 0.0663 - val_loss: 0.3143 - val_output_Central_loss: 0.0788 - val_output_East_loss: 0.0519 - val_output_North_loss: 0.0433 - val_output_South_loss: 0.0766 - val_output_West_loss: 0.0637
Epoch 3/20
5592/5592 ━━━━━━━━━━━━━━━━━━━━ 23s 4ms/step - loss: 0.3334 - output_Central_loss: 0.0883 - output_East_loss: 0.0522 - output_North_loss: 0.0449 - output_South_loss: 0.0868 - output_West_loss: 0.0613 - val_loss: 

In [20]:
# Dictionary to store evaluation results for each region
evaluation_results = {}

# Make predictions on the test set
predictions = model.predict(X_test)

# Ensure predictions is a list, even if there's only one output
if not isinstance(predictions, list):
    predictions = [predictions]

# Iterate through each region to evaluate performance
for i, region in enumerate(region_names):
    # Extract true values for the current region from y_test_mtl
    y_true_region = y_test_mtl[f'output_{region}']

    # Extract predicted values for the current region
    y_pred_region = predictions[i]

    # Flatten the arrays to 1D for easier filtering and metric calculation
    y_true_region_flat = y_true_region.flatten()
    y_pred_region_flat = y_pred_region.flatten()

    # Create a mask to filter out placeholder values
    valid_mask = y_true_region_flat != PLACEHOLDER_VALUE

    # Apply the mask to get only valid true and predicted values
    y_true_valid = y_true_region_flat[valid_mask]
    y_pred_valid = y_pred_region_flat[valid_mask]


    # Calculate metrics
    mse = mean_squared_error(y_true_valid, y_pred_valid)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_valid, y_pred_valid)

    evaluation_results[region] = {
            'MSE': mse,
            'RMSE': rmse,
            'R-squared': r2
        }

    print('Neural Network Results for Region:', region)
    print('Mean Squared Error (MSE): ', mse)
    print('Root Mean Squared Error (RMSE): ', rmse)
    print('R-squared: ', round(r2 * 100, 2))
    print()

177/177 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Neural Network Results for Region: Central
Mean Squared Error (MSE):  0.07833635061979294
Root Mean Squared Error (RMSE):  0.279886317314357
R-squared:  94.7

Neural Network Results for Region: East
Mean Squared Error (MSE):  0.042885009199380875
Root Mean Squared Error (RMSE):  0.20708696047646474
R-squared:  92.81

Neural Network Results for Region: North
Mean Squared Error (MSE):  0.04715332016348839
Root Mean Squared Error (RMSE):  0.21714815256752332
R-squared:  89.22

Neural Network Results for Region: South
Mean Squared Error (MSE):  0.09519921988248825
Root Mean Squared Error (RMSE):  0.3085437082205506
R-squared:  94.56

Neural Network Results for Region: West
Mean Squared Error (MSE):  0.051764506846666336
Root Mean Squared Error (RMSE):  0.22751814619204846
R-squared:  91.58



In [21]:
# Variables to store best the model's weighted metrics, batch size and predictions
best_mse = 0
best_r2 = 0
best_batch_size = 0
best_model_predictions = None

# Calculating Weighted Metrics
# using Weighted_metric = sum(Region metric * (Test samples in region/Total samples in test set))

region_samples = {}
for region in region_names:
    y_true_region = y_test_mtl[f'output_{region}'].flatten()
    valid_mask = y_true_region != PLACEHOLDER_VALUE
    region_samples[region] = np.sum(valid_mask)

# Filter out regions with no valid samples to avoid division by zero
filtered_regions = {region: sample for region, sample in region_samples.items() if sample > 0}

# Calculate total samples
total_samples = sum(filtered_regions.values())

weighted_mse = 0
weighted_r2 = 0

for region, sample in filtered_regions.items():
    weight = sample / total_samples
    weighted_mse = weighted_mse + evaluation_results[region]['MSE'] * weight
    weighted_r2 = weighted_r2 + evaluation_results[region]['R-squared'] * weight

best_mse = weighted_mse
best_r2 = weighted_r2
best_batch_size = 32
best_model_predictions = predictions

print('MTL Approach Results: ')
print('Mean Squared Error (MSE): ', weighted_mse)
print('Root Mean Squared Error (RMSE): ', np.sqrt(weighted_mse))
print('R-squared: ', round(weighted_r2 * 100, 2))

MTL Approach Results: 
Mean Squared Error (MSE):  0.0567538870899845
Root Mean Squared Error (RMSE):  0.23823074337705555
R-squared:  91.86


### Tuning

The initial models' performance are very good, with some performing slightly better than others. We will try tuning the batch size using a larger value.

In [22]:
# Train the model with batch size of 64
history = model.fit(X_train, y_train_mtl,
                    epochs=20,
                    batch_size=64,
                    validation_split=0.2,
                    verbose=1)

# Make predictions on the test set
predictions = model.predict(X_test)

# Ensure predictions is a list, even if there's only one output
if not isinstance(predictions, list):
    predictions = [predictions]

# Iterate through each region to evaluate performance
for i, region in enumerate(region_names):
    # Extract true values for the current region from y_test_mtl
    y_true_region = y_test_mtl[f'output_{region}']

    # Extract predicted values for the current region
    y_pred_region = predictions[i]

    # Flatten the arrays to 1D for easier filtering and metric calculation
    y_true_region_flat = y_true_region.flatten()
    y_pred_region_flat = y_pred_region.flatten()

    # Create a mask to filter out placeholder values
    valid_mask = y_true_region_flat != PLACEHOLDER_VALUE

    # Apply the mask to get only valid true and predicted values
    y_true_valid = y_true_region_flat[valid_mask]
    y_pred_valid = y_pred_region_flat[valid_mask]


    # Calculate metrics
    mse = mean_squared_error(y_true_valid, y_pred_valid)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_valid, y_pred_valid)

    evaluation_results[region] = {
            'MSE': mse,
            'RMSE': rmse,
            'R-squared': r2
        }

# Calculating Weighted Metrics
# using Weighted_metric = sum(Region metric * (Test samples in region/Total samples in test set))

region_samples = {}
for region in region_names:
    y_true_region = y_test_mtl[f'output_{region}'].flatten()
    valid_mask = y_true_region != PLACEHOLDER_VALUE
    region_samples[region] = np.sum(valid_mask)

# Filter out regions with no valid samples to avoid division by zero
filtered_regions = {region: sample for region, sample in region_samples.items() if sample > 0}

# Calculate total samples
total_samples = sum(filtered_regions.values())

weighted_mse = 0
weighted_r2 = 0

for region, sample in filtered_regions.items():
    weight = sample / total_samples
    weighted_mse = weighted_mse + evaluation_results[region]['MSE'] * weight
    weighted_r2 = weighted_r2 + evaluation_results[region]['R-squared'] * weight

print('Neural Network Tuning Results: ')
if weighted_mse > best_mse:
    best_mse = weighted_mse
    best_r2 = weighted_r2
    best_batch_size = 64
    best_model_predictions = predictions
    print('New best batch size: 64')
else:
    print('No new best batch size')

Epoch 1/20
2796/2796 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.2621 - output_Central_loss: 0.0674 - output_East_loss: 0.0420 - output_North_loss: 0.0350 - output_South_loss: 0.0687 - output_West_loss: 0.0490 - val_loss: 0.3204 - val_output_Central_loss: 0.0845 - val_output_East_loss: 0.0525 - val_output_North_loss: 0.0542 - val_output_South_loss: 0.0742 - val_output_West_loss: 0.0551
Epoch 2/20
2796/2796 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.2616 - output_Central_loss: 0.0679 - output_East_loss: 0.0424 - output_North_loss: 0.0353 - output_South_loss: 0.0672 - output_West_loss: 0.0488 - val_loss: 0.3523 - val_output_Central_loss: 0.0920 - val_output_East_loss: 0.0567 - val_output_North_loss: 0.0458 - val_output_South_loss: 0.0963 - val_output_West_loss: 0.0615
Epoch 3/20
2796/2796 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.2598 - output_Central_loss: 0.0672 - output_East_loss: 0.0419 - output_North_loss: 0.0349 - output_South_loss: 0.0670 - output_West_loss: 0.0489 - val_loss: 

### Final Model

Using the best performing model's saved results, calculate the weighted RMSE in dollars.

In [23]:
# Get the mean and standard deviation for 'resale_price' from the scaler
resale_price_idx = continuous_cols.index('resale_price')
original_resale_price_mean = scaler.mean_[resale_price_idx]
original_resale_price_std = scaler.scale_[resale_price_idx]


# Temporary variable to store MSE in dollars
weighted_mse_dollars = 0


for i, region in enumerate(region_names):
    if region in filtered_regions:
        current_weight = filtered_regions[region]

        # Get scaled true and predicted values
        y_true_region_scaled = y_test_mtl[f'output_{region}']
        y_pred_region_scaled = best_model_predictions[i] # Use best_model_predictions

        # Flatten and mask to get valid scaled values
        y_true_region_flat_scaled = y_true_region_scaled.flatten()
        y_pred_region_flat_scaled = y_pred_region_scaled.flatten()
        valid_mask = y_true_region_flat_scaled != PLACEHOLDER_VALUE

        y_true_valid_scaled = y_true_region_flat_scaled[valid_mask]
        y_pred_valid_scaled = y_pred_region_flat_scaled[valid_mask]

        # StandardScaler takes the values and subtracts the mean before dividing by the standard deviation
        # To reverse scaling, multiply the scaled values by the standard deviation and then add the mean
        y_true_unscaled = (y_true_valid_scaled * original_resale_price_std) + original_resale_price_mean
        y_pred_unscaled = (y_pred_valid_scaled * original_resale_price_std) + original_resale_price_mean

        # Apply inverse log transformation (np.expm1) to get actual dollar values
        y_true_valid_dollars = np.expm1(y_true_unscaled)
        y_pred_valid_dollars = np.expm1(y_pred_unscaled)

        # Calculate MSE in dollars for this region
        mse_region_dollars = mean_squared_error(y_true_valid_dollars, y_pred_valid_dollars)

        # Accumulate weighted MSE in dollars
        weighted_mse_dollars = weighted_mse_dollars + mse_region_dollars * (current_weight / total_samples)


# Get the MSE, RMSE and R-squared of the best model
weighted_mse = best_mse
weighted_rmse = np.sqrt(weighted_mse)
weighted_r2 = best_r2
weighted_rmse_dollars = np.sqrt(weighted_mse_dollars)

print('MTL Approach Final Results: ')
print('Mean Squared Error (MSE): ', weighted_mse)
print('Root Mean Squared Error (RMSE): ', weighted_rmse)
print('RMSE in dollars: ', weighted_rmse_dollars)
print('R-squared: ', round(weighted_r2 * 100, 2))

MTL Approach Final Results: 
Mean Squared Error (MSE):  0.06557570529466694
Root Mean Squared Error (RMSE):  0.2560775376612852
RMSE in dollars:  69814.094909493
R-squared:  90.73
